<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/rl/09_policy_gradient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R9 方策勾配法:REINFORCE(notebook)

本文:[`09-policy-gradient.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/09-policy-gradient.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

# CartPole:Gymnasium の CartPole-v1 と同じ運動方程式(オイラー法、0.02 秒きざみ)
# 状態 = (台車の位置, 台車の速度, 棒の角度 [rad], 棒の角速度)、行動 0 = 左に押す、1 = 右に押す
# 毎ステップ報酬 +1。棒が 12 度より傾くか、台車が ±2.4 m を出たら terminated、500 ステップで truncated
class CartPole:
    gravity, masscart, masspole, length, force_mag, tau = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02
    total_mass = masscart + masspole
    polemass_length = masspole * length
    theta_limit, x_limit = 12 * 2 * np.pi / 360, 2.4
    def __init__(self, max_steps=500):
        self.max_steps = max_steps
        self.rng = np.random.default_rng()
    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.rng.uniform(-0.05, 0.05, 4)
        self.t = 0
        return self.state.copy(), {}
    def step(self, action):
        x, x_dot, th, th_dot = self.state
        force = self.force_mag if action == 1 else -self.force_mag
        c, s = np.cos(th), np.sin(th)
        temp = (force + self.polemass_length * th_dot**2 * s) / self.total_mass
        th_acc = (self.gravity * s - c * temp) / (self.length * (4.0 / 3.0 - self.masspole * c**2 / self.total_mass))
        x_acc = temp - self.polemass_length * th_acc * c / self.total_mass
        x, x_dot = x + self.tau * x_dot, x_dot + self.tau * x_acc
        th, th_dot = th + self.tau * th_dot, th_dot + self.tau * th_acc
        self.state = np.array([x, x_dot, th, th_dot])
        self.t += 1
        terminated = bool(abs(x) > self.x_limit or abs(th) > self.theta_limit)
        truncated = (not terminated) and self.t >= self.max_steps
        return self.state.copy(), 1.0, terminated, truncated, {}

## 0. 環境:CartPole

ランダムに押すと、棒はすぐに倒れる。

In [ ]:
env = CartPole()
rng = np.random.default_rng(0)
lengths = []
for ep in range(100):
    s, _ = env.reset(seed=ep); done = False; T = 0
    while not done:
        s, r, terminated, truncated, _ = env.step(int(rng.integers(2)))
        T += 1; done = terminated or truncated
    lengths.append(T)
print(f"ランダムな方策:1エピソードの長さ(= 収益)の平均 {np.mean(lengths):.1f}、最大 {np.max(lengths)}(最大は 500)")

## 1. ソフトマックス方策と $\nabla \log \pi$

本文の式 (1) の方策を作る。`W` は形 `(2, 4)`(行動の数 × 状態の次元)。

**手計算**:本文の確認課題1を解く。

- 手計算:(ここに書く)

**実装**:本文の式 (4) の $\nabla_W \log \pi(a \mid s)$ を、形 `(2, 4)` の配列で返す関数を書く。`np.outer` が使える。

In [ ]:
def softmax(x):
    x = x - np.max(x)                 # 大きな数で exp があふれないように、最大値を引いておく(結果は変わらない)
    e = np.exp(x)
    return e / e.sum()

def policy(W, s):
    return softmax(W @ s)

def grad_log_pi(W, s, a):
    raise NotImplementedError("TODO: one_hot(a) - policy(W, s) と s の外積(np.outer)")

In [ ]:
W1 = np.array([[0.0, 0.0], [1.0, 0.0]]); s1 = np.array([1.0, 2.0])
p1 = 1 / (1 + np.exp(-1))
check("確認課題1:pi(1|s)", policy(W1, s1)[1], p1)
check("確認課題1:grad log pi(1|s)", grad_log_pi(W1, s1, 1), [[-(1 - p1), -2 * (1 - p1)], [1 - p1, 2 * (1 - p1)]])

# 数値微分で確かめる(M3)
rng = np.random.default_rng(0)
W = rng.normal(size=(2, 4)); s = rng.normal(size=4)
for a in [0, 1]:
    num = np.zeros_like(W); h = 1e-6
    for i in range(2):
        for j in range(4):
            Wp, Wm = W.copy(), W.copy(); Wp[i, j] += h; Wm[i, j] -= h
            num[i, j] = (np.log(policy(Wp, s)[a]) - np.log(policy(Wm, s)[a])) / (2 * h)
    check(f"行動 {a}:式 (4) = 数値微分", grad_log_pi(W, s, a), num, tol=1e-7)

## 2. 方策勾配の推定を確かめる(1ステップの場合)

本文の式 (2):$\nabla J = \mathbb{E}[r \nabla \log \pi(a)]$ を確かめる。状態が1つで、行動が3つ、報酬の期待値が $q = (1, 2, 4)$ のバンディット(R1)を考える。方策は $\pi = \mathrm{softmax}(\theta)$ とする。

- **正確な勾配**:$J = \sum_a \pi(a) q(a)$ を、数値微分する
- **推定**:方策に従って行動を選び、報酬(期待値 $q(a)$、標準偏差 1)をもらって、 $r \nabla \log \pi(a)$ を計算する。これを何回もくり返して平均する

**予想**:推定に使う回数を 10 回、1000 回、100000 回と増やすと、推定と正確な勾配の差はどうなるか。

- 予想:(ここに書く)

In [ ]:
q = np.array([1.0, 2.0, 4.0])
theta = np.array([0.5, 0.0, -0.5])
J = lambda th: softmax(th) @ q
exact = np.array([(J(theta + h) - J(theta - h)) / 2e-6 for h in np.eye(3) * 1e-6])
print("正確な勾配:", exact.round(4))

rng = np.random.default_rng(1)
for n in [10, 1000, 100000]:
    pi = softmax(theta)
    a = rng.choice(3, size=n, p=pi)
    r = rng.normal(q[a], 1.0)
    grads = (np.eye(3)[a] - pi) * r[:, None]                    # 1ステップの場合の式 (4):(one_hot - pi) * r
    print(f"n = {n:6d}:推定 {grads.mean(axis=0).round(4)}   差の大きさ {np.linalg.norm(grads.mean(axis=0) - exact):.4f}")

回数を増やすと、推定は正確な勾配に近づく(差は、およそ $1/\sqrt{n}$ で小さくなる。M2 §3)。REINFORCE は、この推定を、1エピソードごとに1回だけ作って使う。だから、1回ごとの推定はとてもばらつく。

## 3. 収益を後ろから計算する

$G_t = r_{t+1} + \gamma G_{t+1}$(本文 §4)を使い、エピソードの最後から順に計算する。前から1つずつ和をとると、エピソードの長さの2乗に比例する時間がかかる。

In [ ]:
def compute_returns(rewards, gamma):
    G = np.zeros(len(rewards))
    raise NotImplementedError("TODO: 後ろから g = rewards[t] + gamma * g を計算して G[t] に入れる")
    return G

In [ ]:
check("確認課題2:gamma=0.9 で報酬 1,1,1", compute_returns([1.0, 1.0, 1.0], 0.9), [2.71, 1.9, 1.0])
check("gamma=1 なら、残りの報酬の和", compute_returns([1.0, 2.0, 3.0], 1.0), [6.0, 5.0, 3.0])

## 4. REINFORCE

本文 §4 の手順を実装する。`use_baseline=True` なら、収益を標準化(平均を引いて、標準偏差で割る)してから使う(本文 §5)。

**実装**:`TODO` の1行(各時刻の勾配上昇)を書く。

In [ ]:
def run_reinforce(n_episodes=800, alpha=0.01, gamma=0.99, seed=0, use_baseline=True):
    rng = np.random.default_rng(seed)
    env = CartPole(); env.reset(seed=seed)
    W = rng.normal(size=(2, 4)) * 0.01
    history = []
    for ep in range(n_episodes):
        states, actions, rewards = [], [], []
        s, _ = env.reset(); done = False
        while not done:
            a = int(rng.choice(2, p=policy(W, s)))
            s2, r, terminated, truncated, _ = env.step(a)
            states.append(s); actions.append(a); rewards.append(r)
            s, done = s2, terminated or truncated
        G = compute_returns(rewards, gamma)
        if use_baseline:
            G = (G - G.mean()) / (G.std() + 1e-8)
        for s_t, a_t, G_t in zip(states, actions, G):
            raise NotImplementedError("TODO: W を alpha * G_t * grad_log_pi(W, s_t, a_t) だけ動かす(勾配上昇)")
        history.append(sum(rewards))
    return W, np.array(history)

In [ ]:
def moving_average(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")

results = [run_reinforce(seed=k) for k in range(3)]         # 3つの seed(1分弱かかる)
for k, (W_k, h) in enumerate(results):
    plt.plot(moving_average(h), label=f"seed {k}")
    print(f"seed {k}:最初の50エピソードの平均 {h[:50].mean():6.1f} → 最後の50エピソードの平均 {h[-50:].mean():6.1f}")
plt.xlabel("episode"); plt.ylabel("return (moving average 20)"); plt.legend(); plt.title("REINFORCE on CartPole"); plt.show()
check("どの seed でも、最後の50エピソードは最初の50エピソードより長く立つ", [h[-50:].mean() > h[:50].mean() for _, h in results], [True] * 3)

In [ ]:
W_learned = results[0][0]
print("学習した W(行 = 行動 0:左、行動 1:右。列 = 位置、速度、角度、角速度):")
print(W_learned.round(2))
print("右に押す好ましさ - 左に押す好ましさ:", (W_learned[1] - W_learned[0]).round(2))

学習した $W$ で、「右に押す好ましさ − 左に押す好ましさ」の、棒の角度と角速度の係数は正になっているはずである。棒が右に傾いている(角度が正)、または右に倒れつつある(角速度が正)ときは、台車を右に押して、棒の下を支えにいく。

## 5. ベースラインは、ばらつきを小さくする

本文 §5 の式 (5) を確かめる。方策を1つに固定して、エピソードを何回も動かし、1エピソードから作った勾配の推定 $\sum_t (G_t - b) \nabla \log \pi$ を、 $b = 0$ (ベースラインなし)と $b$ = 収益の平均 で比べる。

**予想**:2つの推定の平均は、ほぼ同じになるか。ばらつき(分散)はどうか。

- 予想:(ここに書く)

In [ ]:
W_fixed = np.array([[0.0, 0.0, -0.5, -0.2], [0.0, 0.0, 0.5, 0.2]])
rng = np.random.default_rng(0)
env = CartPole(); env.reset(seed=0)

def one_episode(W):
    states, actions, rewards = [], [], []
    s, _ = env.reset(); done = False
    while not done:
        a = int(rng.choice(2, p=policy(W, s)))
        s2, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r); s, done = s2, terminated or truncated
    return compute_returns(rewards, 1.0), [grad_log_pi(W, st, at) for st, at in zip(states, actions)]

episodes = [one_episode(W_fixed) for _ in range(400)]
b = np.mean([G.mean() for G, _ in episodes[:100]])           # ベースライン:別の100エピソードで求めた収益の平均
g_no = np.array([sum(Gt * g for Gt, g in zip(G, gl)) for G, gl in episodes[100:]])
g_bl = np.array([sum((Gt - b) * g for Gt, g in zip(G, gl)) for G, gl in episodes[100:]])
se = g_no.std(axis=0) / np.sqrt(len(g_no))
print("推定の平均(ベースラインなし):", g_no.mean(axis=0).ravel().round(1))
print("推定の平均(ベースラインあり):", g_bl.mean(axis=0).ravel().round(1))
print("平均の標準誤差(目安)        :", se.ravel().round(1))
print(f"分散の合計:ベースラインなし {g_no.var(axis=0).sum():.0f}   ベースラインあり {g_bl.var(axis=0).sum():.0f}")

## 6. 条件を変える

1. ステップサイズ `alpha` を 0.001、0.01、0.1 に変える
2. 割引率 `gamma` を 0.9、0.99、1.0 に変える
3. 1つの条件ごとに seed を 3 つ使い、最後の50エピソードの平均で比べる(少し時間がかかる)

In [ ]:
for alpha in [0.001, 0.01, 0.1]:
    last = [run_reinforce(n_episodes=400, alpha=alpha, seed=k)[1][-50:].mean() for k in range(3)]
    print(f"1. alpha = {alpha:5}: 最後の50エピソードの平均 {np.round(last, 1)}")
for gamma in [0.9, 0.99, 1.0]:
    last = [run_reinforce(n_episodes=400, gamma=gamma, seed=k)[1][-50:].mean() for k in range(3)]
    print(f"2. gamma = {gamma:4}: 最後の50エピソードの平均 {np.round(last, 1)}")

## 7. 研究で使うときは

`cpai_lab` には、方策勾配法は入っていない(本文 §7)。研究では、PyTorch で書かれたライブラリの PPO などを使うことが多い。そのとき、この notebook の次のものが、ライブラリの中でも使われている。

| この notebook | ライブラリの中 |
|---|---|
| ソフトマックス方策の $W$ | ニューラルネットの方策(出力がソフトマックス、または正規分布) |
| `grad_log_pi` を手で書いた | 自動微分(`loss.backward()`) |
| 収益の標準化 | アドバンテージの標準化 |
| ベースライン = 収益の平均 | ベースライン = 学習した価値関数 $V(s)$(actor-critic) |

## 8. 説明(自分の言葉で書く)

1. 方策勾配の式で、「収益 × $\nabla \log \pi$ 」は何をしているか。収益が大きかった行動と小さかった行動で、それぞれ何が起きるか
   - (ここに書く)
2. ベースラインを引いても勾配の期待値が変わらないのはなぜか。それでも引くのはなぜか
   - (ここに書く)
3. 学習の結果を比べるとき、seed を複数使うのはなぜか。この notebook で、seed による違いはどれくらいあったか
   - (ここに書く)